### Removing duplicate triggers

In [1]:
import mne
import numpy as np

# -----------------------------
# FUNCTION: CLEAN STIM CHANNEL
# -----------------------------
def clean_stim_channel_rising_edge(raw, stim_name, duplicate_gap):
    """
    Removes duplicate triggers from a stim channel using rising-edge detection.
    Only modifies the specified stim channel.
    """
    if stim_name not in raw.ch_names:
        print(f"Channel '{stim_name}' not found.")
        return

    ch_idx = raw.ch_names.index(stim_name)
    data = raw._data[ch_idx]

    # Find rising edges: 0 -> >0
    edges = np.where(np.diff(data.astype(int)) > 0)[0] + 1

    if len(edges) == 0:
        print(f"No triggers found in {stim_name}.")
        return

    keep_edges = [edges[0]]  # always keep first rising edge

    # Remove duplicates that occur too close
    for e in edges[1:]:
        if e - keep_edges[-1] > duplicate_gap:
            keep_edges.append(e)
        else:
            # Zero out duplicate
            data[e] = 0

    raw._data[ch_idx] = data
    print(f"{stim_name}: {len(edges) - len(keep_edges)} duplicate triggers removed, {len(keep_edges)} kept.")

# --------------------------------
# FUNCTION: CLEAN RAW OBJECT AND RESAVE
# --------------------------------

def remove_duplicate_triggers(preprocfile, left_channel_name, right_channel_name, duplicate_gap, output_file):
    print("Loading FIF file...")
    raw = mne.io.read_raw_fif(preprocfile, preload=True)

    left_events_before = mne.find_events(raw, stim_channel=left_channel_name)
    right_events_before = mne.find_events(raw, stim_channel=right_channel_name)
    print(f"Before cleaning → Left: {len(left_events_before)}, Right: {len(right_events_before)}")

    clean_stim_channel_rising_edge(raw, left_channel_name, duplicate_gap)
    clean_stim_channel_rising_edge(raw, right_channel_name, duplicate_gap)

    left_events_after = mne.find_events(raw, stim_channel=left_channel_name)
    right_events_after = mne.find_events(raw, stim_channel=right_channel_name)
    print(f"After cleaning → Left: {len(left_events_after)}, Right: {len(right_events_after)}")

    sfreq = raw.info['sfreq']

    all_events = np.vstack([left_events_after, right_events_after])
    descriptions = ([left_channel_name] * len(left_events_after)) + ([right_channel_name] * len(right_events_after))

    # Sort by time
    sort_idx = np.argsort(all_events[:, 0])
    all_events = all_events[sort_idx]
    descriptions = np.array(descriptions)[sort_idx]

    onsets = all_events[:, 0] / sfreq
    durations = np.zeros(len(all_events))
    annotations = mne.Annotations(onsets, durations, descriptions)
    raw.set_annotations(annotations)

    raw.save(output_file, overwrite=True)
    print(f"Saved cleaned file to: {output_file}")


In [3]:
#subjects and sessions for opms
base = "/rdrives/DRS-foundation-brain/zoe_data/BIDS"
deriv = f"{base}/derivatives_anna_01042026"
subjects = ['001', '002', '003', '004', '005', '006', '007', '009', '010', '011', '012', '013', '014', '015', '016']
sessions = ["003", "004"]
tasks = ["braille"]
runs = ["001"]

#OPMS
for subject in subjects:
    for session in sessions:
        for task in tasks:
            for run in runs:

                id = f"sub-{subject}_ses-{session}_task-{task}_run-{run}"
                preproc_file = f"{deriv}/preprocessed/{id}/{id}_preproc-raw.fif"
                output_file = preproc_file.replace("-raw.fif", "-raw_cleaned.fif")

                remove_duplicate_triggers(preproc_file, 'Left_trial', 'Right_trial', 1000, output_file)


                

Loading FIF file...
Opening raw data file /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-003_task-braille_run-001/sub-001_ses-003_task-braille_run-001_preproc-raw.fif...
    Read 5 compensation matrices
    Range : 17069 ... 337499 =     68.276 ...  1349.996 secs
Ready.
Current compensation grade : 3
Reading 0 ... 320430  =      0.000 ...  1281.720 secs...
Finding events on: Left_trial
40 events found on stim channel Left_trial
Event IDs: [1]
Finding events on: Right_trial
39 events found on stim channel Right_trial
Event IDs: [1]
Before cleaning → Left: 40, Right: 39
Left_trial: 0 duplicate triggers removed, 40 kept.
Right_trial: 0 duplicate triggers removed, 39 kept.
Finding events on: Left_trial
40 events found on stim channel Left_trial
Event IDs: [1]
Finding events on: Right_trial
39 events found on stim channel Right_trial
Event IDs: [1]
After cleaning → Left: 40, Right: 39
Writing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivat

/tmp/ipykernel_528053/3514713555.py:71: RuntimeWarning: Omitted 4 annotation(s) that were outside data range.
  raw.set_annotations(annotations)
/tmp/ipykernel_528053/3514713555.py:73: RuntimeWarning: This filename (/rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-003_task-braille_run-001/sub-001_ses-003_task-braille_run-001_preproc-raw_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(output_file, overwrite=True)


Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-003_task-braille_run-001/sub-001_ses-003_task-braille_run-001_preproc-raw_cleaned.fif
[done]
Saved cleaned file to: /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-003_task-braille_run-001/sub-001_ses-003_task-braille_run-001_preproc-raw_cleaned.fif
Loading FIF file...
Opening raw data file /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-004_task-braille_run-001/sub-001_ses-004_task-braille_run-001_preproc-raw.fif...
    Read 5 compensation matrices
    Range : 2573 ... 330092 =     10.292 ...  1320.368 secs
Ready.
Current compensation grade : 3
Reading 0 ... 327519  =      0.000 ...  1310.076 secs...
Finding events on: Left_trial
41 events found on stim channel Left_trial
Event IDs: [1]
Finding events on: Right_trial
40 events found on stim channel Right_trial
Event IDs: [1]
Before cleaning 

/tmp/ipykernel_528053/3514713555.py:73: RuntimeWarning: This filename (/rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-004_task-braille_run-001/sub-001_ses-004_task-braille_run-001_preproc-raw_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(output_file, overwrite=True)


Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-004_task-braille_run-001/sub-001_ses-004_task-braille_run-001_preproc-raw_cleaned.fif
[done]
Saved cleaned file to: /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-001_ses-004_task-braille_run-001/sub-001_ses-004_task-braille_run-001_preproc-raw_cleaned.fif
Loading FIF file...
Opening raw data file /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-002_ses-003_task-braille_run-001/sub-002_ses-003_task-braille_run-001_preproc-raw.fif...
    Read 5 compensation matrices
    Range : 1409 ... 328855 =      5.636 ...  1315.420 secs
Ready.
Current compensation grade : 3
Reading 0 ... 327446  =      0.000 ...  1309.784 secs...
Finding events on: Left_trial
41 events found on stim channel Left_trial
Event IDs: [1]
Finding events on: Right_trial
40 events found on stim channel Right_trial
Event IDs: [1]
Before cleaning 

...

Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-015_ses-004_task-braille_run-001/sub-015_ses-004_task-braille_run-001_preproc-raw_cleaned.fif
[done]
Saved cleaned file to: /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-015_ses-004_task-braille_run-001/sub-015_ses-004_task-braille_run-001_preproc-raw_cleaned.fif
Loading FIF file...
Opening raw data file /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-003_task-braille_run-001/sub-016_ses-003_task-braille_run-001_preproc-raw.fif...
    Read 5 compensation matrices
    Range : 1937 ... 329575 =      7.748 ...  1318.300 secs
Ready.
Current compensation grade : 3
Reading 0 ... 327638  =      0.000 ...  1310.552 secs...
Finding events on: Left_trial
41 events found on stim channel Left_trial
Event IDs: [1]
Finding events on: Right_trial
41 events found on stim channel Right_trial
Event IDs: [1]
Before cleaning 

/tmp/ipykernel_528053/3514713555.py:73: RuntimeWarning: This filename (/rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-003_task-braille_run-001/sub-016_ses-003_task-braille_run-001_preproc-raw_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(output_file, overwrite=True)


Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-003_task-braille_run-001/sub-016_ses-003_task-braille_run-001_preproc-raw_cleaned.fif
[done]
Saved cleaned file to: /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-003_task-braille_run-001/sub-016_ses-003_task-braille_run-001_preproc-raw_cleaned.fif
Loading FIF file...
Opening raw data file /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-004_task-braille_run-001/sub-016_ses-004_task-braille_run-001_preproc-raw.fif...
    Read 5 compensation matrices
    Range : 2046 ... 329765 =      8.184 ...  1319.060 secs
Ready.
Current compensation grade : 3
Reading 0 ... 327719  =      0.000 ...  1310.876 secs...
Finding events on: Left_trial
41 events found on stim channel Left_trial
Event IDs: [1]
Finding events on: Right_trial
40 events found on stim channel Right_trial
Event IDs: [1]
Before cleaning 

/tmp/ipykernel_528053/3514713555.py:73: RuntimeWarning: This filename (/rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-004_task-braille_run-001/sub-016_ses-004_task-braille_run-001_preproc-raw_cleaned.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(output_file, overwrite=True)


Closing /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-004_task-braille_run-001/sub-016_ses-004_task-braille_run-001_preproc-raw_cleaned.fif
[done]
Saved cleaned file to: /rdrives/DRS-foundation-brain/zoe_data/BIDS/derivatives_anna_01042026/preprocessed/sub-016_ses-004_task-braille_run-001/sub-016_ses-004_task-braille_run-001_preproc-raw_cleaned.fif
